# Day 4 — Stored Procedures, Functions, Cursors, Transactions and Dynamic SQL

**Target:** 3–4 years experience SQL hands-on practice.

**Database:** MySQL 8.x. The tables/data are created in a separate setup SQL file; this notebook focuses on syllabus, commands, explanations, practice queries, assignments, and solutions.


## Learning method

1. Run each query manually in MySQL Workbench/CLI.
2. Before executing, predict the result.
3. Change one condition and rerun.
4. Check execution plans where applicable.
5. Complete assignments without looking at the solution first.


## Important execution commands

```sql
SHOW DATABASES;
USE company_sql_practice;
SHOW TABLES;
SELECT DATABASE();
DESCRIBE employees;
```

The setup file contains the five-table schema and large sample dataset.


## Syllabus

- Stored procedure fundamentals
- IN, OUT and INOUT parameters
- Stored functions
- Local variables and DECLARE
- IF / ELSEIF / ELSE
- CASE inside stored programs
- WHILE, REPEAT and LOOP
- Cursor lifecycle: DECLARE, OPEN, FETCH, CLOSE
- NOT FOUND handler
- Cursor-based business processing
- Transactions: START TRANSACTION, COMMIT, ROLLBACK
- SAVEPOINT
- Error handlers
- Dynamic SQL with PREPARE / EXECUTE
- Temporary tables
- Procedure testing and debugging patterns


## Practice Queries

Each query is intentionally written as a workplace-style example. Read the explanation before changing it.


### Practice 1

**What to learn:** Identify tables, filters, grouping/window logic, and expected row grain before execution.


In [ ]:
-- Procedure: employees by department
DELIMITER //
CREATE PROCEDURE get_employees_by_department(IN p_department_id INT)
BEGIN
    SELECT employee_id, employee_name, salary, hire_date
    FROM employees
    WHERE department_id = p_department_id
    ORDER BY salary DESC;
END//
DELIMITER ;
CALL get_employees_by_department(1);


**Check:** Verify row count, NULL behavior, duplicates, and ordering. Explain why the result is correct before moving on.


### Practice 2

**What to learn:** Identify tables, filters, grouping/window logic, and expected row grain before execution.


In [ ]:
-- Function: annual salary
DELIMITER //
CREATE FUNCTION annual_salary(p_monthly_salary DECIMAL(12,2))
RETURNS DECIMAL(14,2)
DETERMINISTIC
BEGIN
    RETURN p_monthly_salary * 12;
END//
DELIMITER ;
SELECT employee_name, salary, annual_salary(salary) AS annual_salary
FROM employees;


**Check:** Verify row count, NULL behavior, duplicates, and ordering. Explain why the result is correct before moving on.


### Practice 3

**What to learn:** Identify tables, filters, grouping/window logic, and expected row grain before execution.


In [ ]:
-- Cursor pattern: process employees into an audit table
DELIMITER //
CREATE PROCEDURE build_salary_audit()
BEGIN
    DECLARE done INT DEFAULT 0;
    DECLARE v_employee_id INT;
    DECLARE v_salary DECIMAL(12,2);

    DECLARE cur CURSOR FOR
        SELECT employee_id, salary FROM employees;
    DECLARE CONTINUE HANDLER FOR NOT FOUND SET done = 1;

    OPEN cur;

    read_loop: LOOP
        FETCH cur INTO v_employee_id, v_salary;
        IF done = 1 THEN
            LEAVE read_loop;
        END IF;

        INSERT INTO salary_audit(employee_id, salary_snapshot, captured_at)
        VALUES (v_employee_id, v_salary, CURRENT_TIMESTAMP);
    END LOOP;

    CLOSE cur;
END//
DELIMITER ;


**Check:** Verify row count, NULL behavior, duplicates, and ordering. Explain why the result is correct before moving on.


### Practice 4

**What to learn:** Identify tables, filters, grouping/window logic, and expected row grain before execution.


In [ ]:
-- Transaction practice
START TRANSACTION;

UPDATE employees
SET salary = salary * 1.05
WHERE department_id = 1;

SAVEPOINT after_dept1;

UPDATE employees
SET salary = salary * 1.02
WHERE department_id = 2;

-- Inspect before deciding
SELECT department_id, SUM(salary) AS payroll
FROM employees
GROUP BY department_id;

ROLLBACK TO SAVEPOINT after_dept1;
COMMIT;


**Check:** Verify row count, NULL behavior, duplicates, and ordering. Explain why the result is correct before moving on.


### Practice 5

**What to learn:** Identify tables, filters, grouping/window logic, and expected row grain before execution.


In [ ]:
-- Dynamic SQL pattern
SET @sql = 'SELECT employee_id, employee_name, salary FROM employees WHERE salary >= ?';
SET @min_salary = 100000;
PREPARE stmt FROM @sql;
EXECUTE stmt USING @min_salary;
DEALLOCATE PREPARE stmt;


**Check:** Verify row count, NULL behavior, duplicates, and ordering. Explain why the result is correct before moving on.


## Assignments

1. Write a procedure returning employees hired between two dates.
2. Write a procedure with an OUT parameter returning department payroll.
3. Write a function that classifies salary into a band.
4. Write a cursor procedure that inserts high-salary employees into an audit table.
5. Demonstrate COMMIT and ROLLBACK with an order transaction.
6. Create an error handler for duplicate-key or SQL exceptions.
7. Explain when a set-based query should replace a cursor.


## Solutions

Try the assignment first. Then compare with the reference pattern below. Adapt names/filters to your dataset.


### Solution 1


In [ ]:
DELIMITER //
CREATE PROCEDURE employees_between_dates(IN p_start DATE, IN p_end DATE)
BEGIN
 SELECT employee_id,employee_name,hire_date
 FROM employees
 WHERE hire_date BETWEEN p_start AND p_end;
END//
DELIMITER ;
CALL employees_between_dates('2022-01-01','2025-12-31');


### Solution 2


In [ ]:
DELIMITER //
CREATE PROCEDURE department_payroll(IN p_department_id INT, OUT p_payroll DECIMAL(14,2))
BEGIN
 SELECT COALESCE(SUM(salary),0) INTO p_payroll
 FROM employees WHERE department_id=p_department_id;
END//
DELIMITER ;
CALL department_payroll(1,@payroll);
SELECT @payroll;


### Solution 3


In [ ]:
DELIMITER //
CREATE FUNCTION salary_band(p_salary DECIMAL(12,2))
RETURNS VARCHAR(20)
DETERMINISTIC
BEGIN
 RETURN CASE
   WHEN p_salary>=150000 THEN 'Executive'
   WHEN p_salary>=100000 THEN 'Senior'
   WHEN p_salary>=70000 THEN 'Mid'
   ELSE 'Junior'
 END;
END//
DELIMITER ;


### Solution 4


In [ ]:
DELIMITER //
CREATE PROCEDURE audit_high_salary()
BEGIN
 DECLARE done INT DEFAULT 0;
 DECLARE v_id INT; DECLARE v_salary DECIMAL(12,2);
 DECLARE cur CURSOR FOR SELECT employee_id,salary FROM employees WHERE salary>=120000;
 DECLARE CONTINUE HANDLER FOR NOT FOUND SET done=1;
 OPEN cur;
 read_loop: LOOP
  FETCH cur INTO v_id,v_salary;
  IF done=1 THEN LEAVE read_loop; END IF;
  INSERT INTO salary_audit(employee_id,salary_snapshot,captured_at)
  VALUES(v_id,v_salary,CURRENT_TIMESTAMP);
 END LOOP;
 CLOSE cur;
END//
DELIMITER ;


### Solution 5


In [ ]:
START TRANSACTION;
INSERT INTO orders(customer_id,order_date,status) VALUES (1,CURRENT_DATE,'Pending');
SET @new_order_id=LAST_INSERT_ID();
INSERT INTO order_items(order_id,product_id,quantity,unit_price)
VALUES(@new_order_id,1,2,100);
-- COMMIT when validated; ROLLBACK to undo both statements.
ROLLBACK;


### Solution 6


In [ ]:
DELIMITER //
CREATE PROCEDURE safe_demo()
BEGIN
 DECLARE EXIT HANDLER FOR SQLEXCEPTION
 BEGIN
   ROLLBACK;
 END;
 START TRANSACTION;
 UPDATE employees SET salary=salary*1.01 WHERE department_id=1;
 COMMIT;
END//
DELIMITER ;


### Solution 7


In [ ]:
-- Prefer set-based SQL for bulk work:
UPDATE employees
SET salary = salary * 1.05
WHERE department_id = 1;


## Interview Questions & Answers

**Q: WHERE vs HAVING?**  
A: WHERE filters rows before grouping; HAVING filters grouped results.

**Q: RANK vs DENSE_RANK?**  
A: Both assign equal ranks to ties; RANK leaves gaps after ties, DENSE_RANK does not.

**Q: Why can joins create duplicates?**  
A: A one-to-many relationship legitimately creates multiple result rows per parent. First define the intended row grain, then aggregate/deduplicate as required.

**Q: Cursor or set-based SQL?**  
A: Prefer set-based operations for bulk relational transformations; use cursors when row-by-row procedural behavior is genuinely required.

**Q: What does EXPLAIN provide?**  
A: It exposes the optimizer's planned access path, including table access, candidate indexes, selected index, estimated rows and other plan details.


## 5-Day Deliverable Checklist

- [ ] All practice queries executed manually
- [ ] All assignments solved without copying
- [ ] Solutions compared and rewritten
- [ ] At least 10 query variations created
- [ ] At least 5 window-function problems solved
- [ ] At least 3 stored procedures created
- [ ] At least 1 cursor + handler implemented
- [ ] Transactions tested with COMMIT and ROLLBACK
- [ ] EXPLAIN used before/after an index
- [ ] Final capstone query set completed
